# 2 — Freeze Stage 3 manifest and initialization pairing
Stage 2 outcomes are never read. Each reset is repeated, and the canonical named-state fingerprint must match before the 288-row manifest is accepted.

In [ ]:
import csv, os, subprocess
from pathlib import Path
R=Path.home()/"async-vla-latency-bench"; OUT=Path.home()/"stage3"; OUT.mkdir(exist_ok=True); P=Path.home()/"LIBERO-plus"; NATIVE=Path.home()/"stage1-native"; ID=Path.home()/"venv-stage1-id/bin/python"; OOD=Path.home()/"venv-stage1-ood/bin/python"
MAN=OUT/"stage3_manifest.csv"; FROZEN=OUT/"stage3_frozen_horizons.json"; AUDIT=OUT/"stage3_initialization_pairing_audit.csv"
bench=subprocess.run(["git","-C",str(R),"rev-parse","HEAD"],capture_output=True,text=True,check=True).stdout.strip(); plus=subprocess.run(["git","-C",str(P),"rev-parse","HEAD"],capture_output=True,text=True,check=True).stdout.strip()
subprocess.run([str(ID),"-m","async_vla_benchmark.scripts.make_stage3_manifest","--output",str(MAN),"--frozen-horizons",str(FROZEN),"--git-sha",bench,"--lerobot-git-sha","2aba372b4e217cc47db28e0f836859b20d1456c9","--libero-plus-git-sha",plus,"--model-revision","8e174154ef5f6c60a8da12ae99c303d8963138c1"],cwd=R,check=True)
base=os.environ.copy(); base.update({"MUJOCO_GL":"egl","PYOPENGL_PLATFORM":"egl","MPLBACKEND":"Agg"})
subprocess.run([str(ID),"-m","async_vla_benchmark.scripts.resolve_stage3_initializations","--config",str(R/"async_vla_benchmark/configs/stage3.yaml"),"--manifest",str(MAN),"--scene","id","--audit-output",str(AUDIT)],cwd=R,env=base,check=True)
ood=base.copy(); ood.update({"PYTHONPATH":str(P),"MAGICK_HOME":str(NATIVE),"PATH":str(NATIVE/"bin")+os.pathsep+ood.get("PATH",""),"LD_LIBRARY_PATH":str(NATIVE/"lib")+os.pathsep+ood.get("LD_LIBRARY_PATH",""),"STAGE1_NATIVE_REEXEC":str(NATIVE)})
subprocess.run([str(OOD),"-m","async_vla_benchmark.scripts.resolve_stage3_initializations","--config",str(R/"async_vla_benchmark/configs/stage3.yaml"),"--manifest",str(MAN),"--scene","ood","--audit-output",str(AUDIT)],cwd=R,env=ood,check=True)


In [ ]:
rows=list(csv.DictReader(open(MAN))); audit=list(csv.DictReader(open(AUDIT)))
assert len(rows)==288 and len({r['run_id'] for r in rows})==288; assert {r['configured_n_action_steps'] for r in rows}=={'20','25','30'}; assert {r['added_delay_ms'] for r in rows}=={'0','200'}; assert {r['seed'] for r in rows}=={str(x) for x in range(14,22)}
assert len(audit)==48 and all(r['repeatability_pass']=='True' for r in audit); assert {r['fingerprint_method'] for r in audit}=={'mujoco_reset_state_v2_sha256'}; assert all(r['fingerprint']==r['repeat_fingerprint'] for r in audit)
assert {r['canonical_schema'] for r in audit}=={'qpos,qvel,act,ctrl,mocap_pos,mocap_quat; float64 little-endian; round=1e-12; excludes sim time'}
subprocess.run([str(OOD),"-m","async_vla_benchmark.scripts.validate_stage3","--manifest",str(MAN),"--output-dir",str(OUT),"--allow-incomplete"],cwd=R,env=ood,check=True)
print("PASS: 48 reset identities repeated exactly; versioned named-state schema excludes time/serialization noise")
print("PASS: frozen 288-row manifest, exact variants, all six-cell pairing blocks resolved")
print("STOP HERE and paste both PASS lines before notebook 03.")
